# 📊 Phase 1–3 — Dataset Analysis, Manifold Visualization & Baseline Benchmarks

**Project**: Low-Resource Language (Ukrainian) Sentiment & Emotion Analysis  
**Dataset**: [KSE-RESEARCH-Group/UAReviews](https://huggingface.co/datasets/KSE-RESEARCH-Group/UAReviews) (11,580 Ukrainian customer reviews)  
**Embeddings**: Pre-encoded with [Qwen/Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) (1024-d dense vectors)  
**Hardware**: Torch-CUDA GPU runtime

### Objectives:
1. **Phase 1 (Data Preprocessing & Split Integrity)**: Load pre-encoded embeddings and verify official benchmark splits (`train`: 8,106, `test`: 1,737, `challenge`: 1,737) with zero data leakage.
2. **Phase 2 (Exploratory Data Analysis)**: Quantify severe class imbalance across the 7 emotion classes (Happiness 65.3% down to Fear 0.5%), inspect intent categories, text lengths, and rating correlations.
3. **Phase 3 (Semantic Manifold & Cluster Visualization)**: Depict the labeled data in embedding space using **UMAP**, **t-SNE**, and **PCA** to visualize natural semantic clusters, compute class centroid geometries, neighborhood purities, and identify manifold decision boundaries.
4. **Phase 3b (Baseline Benchmarks)**: Train baseline `LinearSVC` and `TorchMLP` on raw imbalanced embeddings to document baseline performance and tail-class collapse before oversampling.


In [ ]:
# 1. Install & Imports
!pip install -q scikit-learn matplotlib seaborn umap-learn torch pandas numpy

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix, silhouette_samples, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Publication-grade Matplotlib aesthetic setup (python-data-visualization skill)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "font.size": 8.5,
    "axes.labelsize": 9.5,
    "axes.titlesize": 10.5,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 8.0,
    "figure.titlesize": 12.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
})

# Curated accessible color palette for the 7 Ukrainian review emotion classes
# Slate Blue for dominant head class, distinct non-saturated hues for tail classes
EMOTION_PALETTE = {
    'Happiness': '#345995',  # Slate blue (majority / head)
    'Anger':     '#d95f02',  # Burnt amber
    'Neutral':   '#7570b3',  # Muted purple
    'Sadness':   '#1b9e77',  # Deep teal
    'Disgust':   '#e7298a',  # Dark rose / magenta
    'Surprise':  '#e6ab02',  # Ochre yellow
    'Fear':      '#a6761d',  # Muted warm bronze
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")


In [ ]:
# 2. Mount Drive and Load Artifacts
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/lrl_sentiment_short'
except Exception:
    BASE_DIR = os.path.abspath('./lrl_cache')

CACHE_DIR = os.path.join(BASE_DIR, 'cache')
EMBED_DIR = os.path.join(CACHE_DIR, 'embeddings')
PHASE13_DIR = os.path.join(CACHE_DIR, 'phase1_3')
os.makedirs(PHASE13_DIR, exist_ok=True)

# Load metadata & label mapping
with open(os.path.join(CACHE_DIR, 'label_encoder.json'), 'r', encoding='utf-8') as f:
    label_map = json.load(f)
emotion2id = label_map['emotion2id']
id2emotion = {int(k): v for k, v in label_map['id2emotion'].items()}
EMOTION_LABELS = [id2emotion[i] for i in range(len(id2emotion))]

# Load cleaned dataframe
df = pd.read_parquet(os.path.join(CACHE_DIR, 'ua_reviews_clean.parquet'))

# Load pre-encoded embeddings
X_all = np.load(os.path.join(EMBED_DIR, 'qwen3_embeddings_all.npy'))

with open(os.path.join(CACHE_DIR, 'split_indices.json'), 'r') as f:
    split_indices = json.load(f)

train_idx = np.array(split_indices['train'])
test_idx = np.array(split_indices['test'])
chal_idx = np.array(split_indices['challenge'])

y_all = df['label'].values
y_train, y_test, y_chal = y_all[train_idx], y_all[test_idx], y_all[chal_idx]
X_train, X_test, X_chal = X_all[train_idx], X_all[test_idx], X_all[chal_idx]

print(f"Dataset successfully loaded:")
print(f"  Total records: {len(df)} | Embeddings: {X_all.shape}")
print(f"  Train:     {X_train.shape[0]} samples ({len(train_idx)/len(df)*100:.1f}%)")
print(f"  Test:      {X_test.shape[0]} samples ({len(test_idx)/len(df)*100:.1f}%)")
print(f"  Challenge: {X_chal.shape[0]} samples ({len(chal_idx)/len(df)*100:.1f}%)")


## 2. Exploratory Data Analysis & Class Imbalance Characterization


In [ ]:
# 3a. Training Set Class Imbalance & Imbalance Ratios
# Imbalance Ratios quantification across the 7 emotion classes
train_counts = pd.Series(y_train).value_counts().sort_index()
maj_count = train_counts.max()
imbalance_table = pd.DataFrame({
    'Emotion': EMOTION_LABELS,
    'Train Count': [train_counts.get(i, 0) for i in range(len(EMOTION_LABELS))],
    'Train %': [train_counts.get(i, 0) / len(y_train) * 100 for i in range(len(EMOTION_LABELS))],
    'Imbalance Ratio (Maj:Class)': [f"{maj_count / max(train_counts.get(i, 1), 1):.1f} : 1" for i in range(len(EMOTION_LABELS))]
})
print("=== Imbalance Ratios in Training Set ===")
print(imbalance_table.to_string(index=False))

# Standalone Figure: Training Set Class Imbalance Distribution
df_sorted = imbalance_table.sort_values(by='Train Count', ascending=True).reset_index(drop=True)
y_pos = np.arange(len(df_sorted))

fig, ax = plt.subplots(figsize=(9.2, 5.0), dpi=300)

bars = ax.barh(
    y_pos,
    df_sorted['Train Count'],
    color=[EMOTION_PALETTE[e] for e in df_sorted['Emotion']],
    height=0.68,
    zorder=3
)

ax.set_yticks(y_pos)
ax.set_yticklabels(df_sorted['Emotion'], fontweight='bold')
ax.set_xlabel("Number of Training Samples (Absolute Count)", fontweight='bold')
ax.set_title("Training Set Class Distribution & Imbalance Severity", fontweight='bold', pad=10)
ax.grid(axis='y', visible=False)
ax.grid(axis='x', visible=True, linestyle=':', alpha=0.5, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Clean dynamic count and percentage labels (no interfering lines)
for bar, count, pct, ratio in zip(bars, df_sorted['Train Count'], df_sorted['Train %'], df_sorted['Imbalance Ratio (Maj:Class)']):
    ir_val = ratio.split(':')[0].strip()
    ax.text(
        bar.get_width() + 60,
        bar.get_y() + bar.get_height() / 2,
        f"{count:,} ({pct:.1f}%) [IR {ir_val}:1]",
        va='center',
        ha='left',
        fontsize=8.0,
        color='#222222'
    )
ax.set_xlim(0, max(df_sorted['Train Count']) * 1.35)

plt.tight_layout()
plt.savefig(os.path.join(PHASE13_DIR, 'class_imbalance_distribution.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(PHASE13_DIR, 'class_imbalance_distribution.pdf'), bbox_inches='tight')
plt.show()


In [ ]:
# 3b. Class Distribution Across Dataset Splits (Stratification)
# Left: Comparison graph with overall average percentage | Right: Table with Split, Emotion, N counts, and Avg %
split_df = df.groupby(['split', 'final_emotion']).size().unstack(fill_value=0)
split_df = split_df.reindex(columns=EMOTION_LABELS)

# Proportions within each split (%)
split_pct = split_df.div(split_df.sum(axis=1), axis=0) * 100
split_pct = split_pct.reindex(columns=df_sorted['Emotion'])
total_all = int(split_df.values.sum())

fig, (ax_plot, ax_table) = plt.subplots(1, 2, figsize=(15.8, 5.4), gridspec_kw={'width_ratios': [1.15, 0.85]}, dpi=300)

# Left Graph: Grouped Horizontal Bar Chart with Average % Annotations
width = 0.25
split_colors = {'train': '#345995', 'test': '#1b9e77', 'challenge': '#d95f02'}

for i, split in enumerate(['train', 'test', 'challenge']):
    ax_plot.barh(
        y_pos + (i - 1) * width,
        split_pct.loc[split],
        height=width * 0.88,
        color=split_colors[split],
        label=f"{split.capitalize()} ({split_df.loc[split].sum():,} samples)",
        zorder=3
    )

# Display overall average percentage next to each grouped emotion bar
for idx, emo in enumerate(df_sorted['Emotion']):
    tot_emo = int(split_df[emo].sum())
    avg_pct = tot_emo / total_all * 100
    max_bar_w = max(split_pct.loc['train', emo], split_pct.loc['test', emo], split_pct.loc['challenge', emo])
    ax_plot.text(
        max_bar_w + 1.2,
        y_pos[idx],
        f"Avg: {avg_pct:.1f}%",
        va='center',
        ha='left',
        fontsize=8.0,
        fontweight='bold',
        color='#2b3e50'
    )

ax_plot.set_yticks(y_pos)
ax_plot.set_yticklabels(df_sorted['Emotion'], fontweight='bold')
ax_plot.set_xlabel("Class Share within Split (%)", fontweight='bold')
ax_plot.set_title("Class Share Across Dataset Splits (with Overall Avg %)", fontweight='bold', pad=10)
ax_plot.grid(axis='y', visible=False)
ax_plot.grid(axis='x', visible=True, linestyle=':', alpha=0.5, zorder=0)
ax_plot.spines['top'].set_visible(False)
ax_plot.spines['right'].set_visible(False)
ax_plot.set_xlim(0, 85)
ax_plot.legend(loc='lower right', frameon=True, facecolor='white', framealpha=0.92)

# Right Table: Sample counts and average percentage by Split and Emotion
ax_table.axis('off')
table_rows = []
# Order from Head to Tail for natural readability
emotions_head_to_tail = df_sorted['Emotion'].tolist()[::-1]
for emo in emotions_head_to_tail:
    tr_n = int(split_df.loc['train', emo])
    te_n = int(split_df.loc['test', emo])
    ch_n = int(split_df.loc['challenge', emo])
    tot_n = tr_n + te_n + ch_n
    avg_p = tot_n / total_all * 100
    table_rows.append([emo, f"{tr_n:,}", f"{te_n:,}", f"{ch_n:,}", f"{tot_n:,}", f"{avg_p:.1f}%"])

train_tot = int(split_df.loc['train'].sum())
test_tot = int(split_df.loc['test'].sum())
chal_tot = int(split_df.loc['challenge'].sum())
table_rows.append(['Total', f"{train_tot:,}", f"{test_tot:,}", f"{chal_tot:,}", f"{total_all:,}", '100.0%'])

tab = ax_table.table(
    cellText=table_rows,
    colLabels=['Emotion', 'Train (N)', 'Test (N)', 'Chal (N)', 'Total (N)', 'Avg (%)'],
    loc='center',
    cellLoc='center'
)
tab.auto_set_font_size(False)
tab.set_fontsize(8.2)
tab.scale(1.0, 1.45)

# Style table headers and totals
for j in range(6):
    tab[0, j].set_facecolor('#2b3e50')
    tab[0, j].set_text_props(color='white', fontweight='bold')
    tab[len(table_rows), j].set_facecolor('#e9ecef')
    tab[len(table_rows), j].set_text_props(fontweight='bold')

# Alternating zebra row shading
for i in range(1, len(table_rows)):
    bg = '#f8f9fa' if i % 2 == 1 else '#ffffff'
    for j in range(6):
        tab[i, j].set_facecolor(bg)

ax_table.set_title("Sample Counts & Overall Share by Split", fontweight='bold', pad=10)

plt.tight_layout(w_pad=2.5)
plt.savefig(os.path.join(PHASE13_DIR, 'split_stratification_distribution.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(PHASE13_DIR, 'split_stratification_distribution.pdf'), bbox_inches='tight')
plt.show()

# Print formatted summary table to console
summary_df = pd.DataFrame(
    table_rows[:-1],
    columns=['Emotion', 'Train (N)', 'Test (N)', 'Chal (N)', 'Total (N)', 'Avg (%)']
)
print("\n=== Sample Counts & Share by Split ===")
print(summary_df.to_string(index=False))


In [ ]:
# 4. Text Length & Rating Distribution Analysis
df['char_length'] = df['content'].str.len()
df['word_count'] = df['content'].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(16.8, 5.0), dpi=300)

# Panel A: Character Length by Emotion (Horizontal Boxplot)
sns.boxplot(
    data=df,
    y='final_emotion',
    x='char_length',
    hue='final_emotion',
    order=EMOTION_LABELS,
    palette=EMOTION_PALETTE,
    legend=False,
    ax=axes[0],
    boxprops=dict(alpha=0.85, edgecolor='#333333', linewidth=1.0),
    medianprops=dict(color='black', linewidth=1.8),
    whiskerprops=dict(color='#555555', linewidth=1.0),
    capprops=dict(color='#555555', linewidth=1.0),
    flierprops=dict(marker='o', markersize=2.5, alpha=0.35, markeredgecolor='none')
)
axes[0].set_xscale('log')
axes[0].set_title("Review Character Length by Emotion", fontweight='bold', pad=8)
axes[0].set_xlabel("Length in Characters (log scale)")
axes[0].set_ylabel("Emotion Class")
axes[0].grid(axis='x', linestyle=':', alpha=0.5, zorder=0)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Panel B: Emotion vs Intent Category (%) Heatmap
cat_ct = pd.crosstab(df['final_emotion'], df['final_category'], normalize='index') * 100
cat_ct = cat_ct.reindex(EMOTION_LABELS)
sns.heatmap(
    cat_ct,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    linewidths=0.8,
    linecolor='white',
    cbar_kws={'label': '% of Emotion Class'},
    annot_kws={'size': 7.5, 'weight': 'semibold'},
    ax=axes[1]
)
axes[1].set_title("Emotion vs Intent Category Distribution", fontweight='bold', pad=8)
axes[1].set_ylabel("")
axes[1].set_xlabel("Review Intent Category")
axes[1].tick_params(axis='x', rotation=30)
axes[1].tick_params(axis='y', rotation=0)

# Panel C: Emotion vs Star Rating (1-5) Heatmap
rating_ct = pd.crosstab(df['final_emotion'], df['rating'].fillna('Missing'), normalize='index') * 100
rating_ct = rating_ct.reindex(EMOTION_LABELS)
sns.heatmap(
    rating_ct,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.8,
    linecolor='white',
    cbar_kws={'label': '% of Emotion Class'},
    annot_kws={'size': 7.5, 'weight': 'semibold'},
    ax=axes[2]
)
axes[2].set_title("Emotion vs Star Rating Correlation", fontweight='bold', pad=8)
axes[2].set_ylabel("")
axes[2].set_xlabel("Star Rating (1–5)")
axes[2].tick_params(axis='y', rotation=0)

plt.tight_layout(w_pad=2.0)
plt.savefig(os.path.join(PHASE13_DIR, 'eda_length_intent_rating.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(PHASE13_DIR, 'eda_length_intent_rating.pdf'), bbox_inches='tight')
plt.show()


## 3. Semantic Manifold & Embedding Space Visualization (Natural Clusters)

To reveal how Ukrainian customer reviews arrange themselves in the 1024-dimensional semantic space of `Qwen3-Embedding-0.6B`, we employ:
1. **UMAP (Uniform Manifold Approximation and Projection)**: Preserves both local and global geometric structure.
2. **t-SNE (t-Distributed Stochastic Neighbor Embedding)**: Focuses on local neighborhood preservation.
3. **PCA (Principal Component Analysis)**: Linear orthogonal projection showing primary variance axes.


In [ ]:
# 5. Dimensionality Reduction: UMAP & t-SNE Projections
import umap
from sklearn.manifold import TSNE

print("Computing UMAP 2D projection on 1024-d embeddings...")
umap_reducer = umap.UMAP(n_neighbors=25, min_dist=0.15, metric='cosine', random_state=42)
X_umap = umap_reducer.fit_transform(X_all)

print("Computing t-SNE 2D projection on 1024-d embeddings...")
pca_50 = PCA(n_components=50, random_state=42).fit_transform(X_all)
tsne_reducer = TSNE(n_components=2, perplexity=35, max_iter=1000, random_state=42, n_jobs=-1)
X_tsne = tsne_reducer.fit_transform(pca_50)

print("Projections computed successfully!")


In [ ]:
# 6. Plot Natural Clusters: UMAP & t-SNE (Publication Layout & Layering)
fig, axes = plt.subplots(1, 2, figsize=(16.0, 6.4), dpi=300)

# Ordering guardrail: majority class first (background), tail classes on top
render_order = list(range(len(EMOTION_LABELS)))

for ax_idx, (ax, X_proj, proj_name) in enumerate([
    (axes[0], X_umap, "UMAP 2D Projection"),
    (axes[1], X_tsne, "t-SNE 2D Projection")
]):
    for label_idx in render_order:
        label = EMOTION_LABELS[label_idx]
        mask = (y_all == label_idx)
        n_pts = int(mask.sum())
        color = EMOTION_PALETTE[label]

        if label == 'Happiness':
            alpha = 0.20
            size = 10
            zorder = 1
            edgecolor = 'none'
        elif label in ['Anger', 'Neutral', 'Sadness']:
            alpha = 0.55
            size = 18
            zorder = 2
            edgecolor = 'none'
        else:  # Tail classes: Disgust, Surprise, Fear
            alpha = 0.85
            size = 28
            zorder = 3
            edgecolor = '#222222'

        ax.scatter(
            X_proj[mask, 0], X_proj[mask, 1],
            label=f"{label} (n={n_pts:,})",
            color=color,
            alpha=alpha,
            s=size,
            zorder=zorder,
            edgecolors=edgecolor,
            linewidths=0.4 if edgecolor != 'none' else 0
        )

    ax.set_title(f"{proj_name} — Qwen3-Embedding-0.6B (1024-d)", fontweight='bold', pad=8)
    ax.set_xlabel(f"{proj_name.split()[0]} Dim 1")
    ax.set_ylabel(f"{proj_name.split()[0]} Dim 2")
    ax.grid(True, linestyle=':', alpha=0.4, zorder=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(
        loc='upper right',
        frameon=True,
        facecolor='white',
        framealpha=0.92,
        edgecolor='#cccccc',
        markerscale=1.8,
        fontsize=7.8,
        title="Emotion Class (Sample Count)",
        title_fontsize=8.2
    )

plt.tight_layout(w_pad=2.0)
plt.savefig(os.path.join(PHASE13_DIR, 'embedding_natural_clusters_umap_tsne.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(PHASE13_DIR, 'embedding_natural_clusters_umap_tsne.pdf'), bbox_inches='tight')
plt.show()


In [ ]:
# 7. PCA Scree Plot (Dual Y-Axis) & 2D Projection
pca_full = PCA(n_components=50, random_state=42).fit(X_all)
evr = pca_full.explained_variance_ratio_
cum_evr = np.cumsum(evr)

fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.4), dpi=300)

# Panel A: Dual-Axis Scree Plot (Individual vs Cumulative Explained Variance)
ax_left = axes[0]
ax_right = ax_left.twinx()

pcs = np.arange(1, 51)
# Primary Y-axis: Individual component variance ratio
bars = ax_left.bar(pcs, evr * 100, color='#345995', alpha=0.75, width=0.65, label='Individual EVR (%)', zorder=2)
ax_left.set_xlabel("Principal Component Index")
ax_left.set_ylabel("Individual Explained Variance (%)", color='#345995', fontweight='semibold')
ax_left.tick_params(axis='y', labelcolor='#345995')
ax_left.set_ylim(0, max(evr * 100) * 1.25)
ax_left.grid(axis='x', linestyle=':', alpha=0.4)

# Secondary Y-axis: Cumulative explained variance ratio
line_cum = ax_right.plot(pcs, cum_evr * 100, color='#d95f02', linewidth=2.0, marker='s', markersize=3.5, label='Cumulative EVR (%)', zorder=3)
ax_right.set_ylabel("Cumulative Explained Variance (%)", color='#d95f02', fontweight='semibold')
ax_right.tick_params(axis='y', labelcolor='#d95f02')
ax_right.set_ylim(0, 105)
ax_right.grid(axis='y', linestyle=':', alpha=0.4)

# Prominent reference lines at 50% and 80% cumulative variance
pc_50 = int(np.argmax(cum_evr >= 0.50)) + 1
pc_80 = int(np.argmax(cum_evr >= 0.80)) + 1
ax_right.axhline(50, color='#555555', linestyle='--', linewidth=0.9, alpha=0.7)
ax_right.axhline(80, color='#555555', linestyle=':', linewidth=0.9, alpha=0.7)
ax_right.text(32, 52, f"50% Cutoff (PC {pc_50})", va='bottom', ha='left', fontsize=7.5, color='#444444')
ax_right.text(32, 82, f"80% Cutoff (PC {pc_80})", va='bottom', ha='left', fontsize=7.5, color='#444444')

ax_left.set_title("PCA Scree Plot (Top 50 Components)", fontweight='bold', pad=8)
ax_left.spines['top'].set_visible(False)
ax_right.spines['top'].set_visible(False)

# Merged clean legend
lines1, labels1 = ax_left.get_legend_handles_labels()
lines2, labels2 = ax_right.get_legend_handles_labels()
ax_left.legend(lines1 + lines2, labels1 + labels2, loc='upper center', frameon=True, facecolor='white', framealpha=0.92)

# Panel B: 2D PCA Scatter Projection
for label_idx in range(len(EMOTION_LABELS)):
    label = EMOTION_LABELS[label_idx]
    mask = (y_all == label_idx)
    n_pts = int(mask.sum())
    color = EMOTION_PALETTE[label]

    if label == 'Happiness':
        alpha = 0.20
        size = 10
        zorder = 1
        edgecolor = 'none'
    elif label in ['Anger', 'Neutral', 'Sadness']:
        alpha = 0.55
        size = 18
        zorder = 2
        edgecolor = 'none'
    else:
        alpha = 0.85
        size = 28
        zorder = 3
        edgecolor = '#222222'

    axes[1].scatter(
        pca_50[mask, 0], pca_50[mask, 1],
        label=f"{label} (n={n_pts:,})",
        color=color,
        alpha=alpha,
        s=size,
        zorder=zorder,
        edgecolors=edgecolor,
        linewidths=0.4 if edgecolor != 'none' else 0
    )

axes[1].set_title("PCA 2D Projection (PC1 vs PC2)", fontweight='bold', pad=8)
axes[1].set_xlabel(f"PC1 ({evr[0]*100:.1f}% Variance)")
axes[1].set_ylabel(f"PC2 ({evr[1]*100:.1f}% Variance)")
axes[1].grid(True, linestyle=':', alpha=0.4, zorder=0)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].legend(
    loc='upper right',
    frameon=True,
    facecolor='white',
    framealpha=0.92,
    edgecolor='#cccccc',
    markerscale=1.8,
    fontsize=7.8,
    title="Emotion Class",
    title_fontsize=8.2
)

plt.tight_layout(w_pad=2.5)
plt.savefig(os.path.join(PHASE13_DIR, 'pca_analysis.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(PHASE13_DIR, 'pca_analysis.pdf'), bbox_inches='tight')
plt.show()


In [ ]:
# 8. Manifold Geometry: Class Centroids Cosine Similarity & Neighborhood Purity
centroids = np.zeros((len(EMOTION_LABELS), X_train.shape[1]), dtype=np.float32)
for i in range(len(EMOTION_LABELS)):
    mask = (y_train == i)
    if mask.sum() > 0:
        c = X_train[mask].mean(axis=0)
        centroids[i] = c / np.linalg.norm(c)

centroid_sim = centroids @ centroids.T
centroid_df = pd.DataFrame(centroid_sim, index=EMOTION_LABELS, columns=EMOTION_LABELS)

# k-NN Neighborhood Purity (k=5)
nbrs = NearestNeighbors(n_neighbors=6, metric='cosine').fit(X_train)
distances, indices = nbrs.kneighbors(X_train)

purities = []
for i in range(len(EMOTION_LABELS)):
    mask = (y_train == i)
    neighbor_labels = y_train[indices[mask, 1:]]
    purity = (neighbor_labels == i).mean()
    purities.append(purity)

purity_df = pd.DataFrame({
    'Emotion': EMOTION_LABELS,
    'Train Count': [int((y_train == i).sum()) for i in range(len(EMOTION_LABELS))],
    '5-NN Purity (%)': [p * 100 for p in purities]
})

fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.8), dpi=300)

# Panel A: Centroid Cosine Similarity Matrix
sns.heatmap(
    centroid_df,
    annot=True,
    fmt=".3f",
    cmap="vlag",
    center=centroid_df.values.mean(),
    square=True,
    linewidths=0.8,
    linecolor='white',
    cbar_kws={'label': 'Cosine Similarity (cos θ)', 'shrink': 0.8},
    annot_kws={'size': 7.8, 'weight': 'semibold'},
    ax=axes[0]
)
axes[0].set_title("Cosine Similarity Between Emotion Centroids (1024-d)", fontweight='bold', pad=10)
axes[0].tick_params(axis='y', rotation=0)
axes[0].tick_params(axis='x', rotation=30)

# Panel B: 5-NN Neighborhood Purity per Class
purity_sorted = purity_df.sort_values(by='5-NN Purity (%)', ascending=True).reset_index(drop=True)
y_pos = np.arange(len(purity_sorted))

bars = axes[1].barh(
    y_pos,
    purity_sorted['5-NN Purity (%)'],
    color=[EMOTION_PALETTE[e] for e in purity_sorted['Emotion']],
    height=0.65,
    zorder=3
)

# Reference Line: Random Chance Baseline (1/7 = 14.29%)
random_chance = 100.0 / len(EMOTION_LABELS)
axes[1].axvline(
    random_chance,
    color='#d62728',
    linestyle='--',
    linewidth=1.5,
    zorder=4,
    label=f'Random Chance Baseline ({random_chance:.1f}%)'
)

axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(purity_sorted['Emotion'], fontweight='bold')
axes[1].set_xlabel("5-NN Neighborhood Purity (%)")
axes[1].set_title("5-NN Local Neighborhood Purity per Class", fontweight='bold', pad=10)
axes[1].set_xlim(0, 105)
axes[1].grid(axis='x', linestyle=':', alpha=0.5, zorder=0)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

for bar, purity, count in zip(bars, purity_sorted['5-NN Purity (%)'], purity_sorted['Train Count']):
    axes[1].text(
        bar.get_width() + 1.5,
        bar.get_y() + bar.get_height() / 2,
        f"{purity:.1f}% (n={count:,})",
        va='center',
        ha='left',
        fontsize=7.8,
        color='#222222',
        fontweight='medium'
    )
axes[1].legend(loc='lower right', frameon=True, facecolor='white', framealpha=0.92)

plt.tight_layout(w_pad=2.5)
plt.savefig(os.path.join(PHASE13_DIR, 'manifold_purity_centroids.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(PHASE13_DIR, 'manifold_purity_centroids.pdf'), bbox_inches='tight')
plt.show()

print("\n--- Neighborhood Purity Table ---")
print(purity_df)


## 4. Baseline Benchmark (Raw Imbalanced Data)

We evaluate two standard classifiers directly on the un-augmented 1024-d embeddings to establish the baseline:
1. `LinearSVC`: Multi-class linear support vector classifier (One-vs-Rest).
2. `TorchMLPClassifier`: GPU-accelerated multi-layer perceptron (Linear -> BatchNorm -> ReLU -> Dropout -> Linear -> BatchNorm -> ReLU -> Dropout -> Linear) trained with Cross-Entropy Loss.


In [ ]:
# 9. PyTorch MLP Classifier Definition
import torch.nn as nn

class TorchMLP(nn.Module):
    def __init__(self, in_dim=1024, hidden_dims=(256, 64), num_classes=7, dropout=0.2):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(d, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            d = h
        layers.append(nn.Linear(d, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.net(x)

def train_torch_mlp(X_tr, y_tr, X_val, y_val, num_classes=7, epochs=60, batch_size=128, lr=1e-3, seed=42):
    torch.manual_seed(seed)
    model = TorchMLP(in_dim=X_tr.shape[1], num_classes=num_classes).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    
    ds_tr = TensorDataset(torch.as_tensor(X_tr, dtype=torch.float32), torch.as_tensor(y_tr, dtype=torch.long))
    loader = DataLoader(ds_tr, batch_size=batch_size, shuffle=True)
    
    X_val_t = torch.as_tensor(X_val, dtype=torch.float32, device=DEVICE)
    y_val_t = torch.as_tensor(y_val, dtype=torch.long, device=DEVICE)
    
    best_f1, best_state = -1, None
    for epoch in range(epochs):
        model.train()
        for bx, by in loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(bx), by)
            loss.backward()
            opt.step()
            
        model.eval()
        with torch.no_grad():
            preds = model(X_val_t).argmax(dim=-1).cpu().numpy()
            val_f1 = f1_score(y_val, preds, average='macro', zero_division=0)
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                
    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict_mlp(model, X):
    model.eval()
    ds = TensorDataset(torch.as_tensor(X, dtype=torch.float32))
    loader = DataLoader(ds, batch_size=256, shuffle=False)
    preds = []
    for (bx,) in loader:
        p = model(bx.to(DEVICE)).argmax(dim=-1).cpu().numpy()
        preds.append(p)
    return np.concatenate(preds)


In [ ]:
# 10. Run Baseline Models on Test and Challenge Sets
def evaluate_preds(y_true, y_pred, split_name, model_name):
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    
    print(f"[{model_name} on {split_name}] Macro F1: {macro_f1:.4f} | Weighted F1: {weighted_f1:.4f} | Acc: {acc:.4f}")
    return {
        'model': model_name,
        'split': split_name,
        'macro_f1': float(macro_f1),
        'weighted_f1': float(weighted_f1),
        'accuracy': float(acc),
        'per_class_f1': {EMOTION_LABELS[i]: float(per_class[i]) for i in range(len(EMOTION_LABELS))}
    }

# 1. LinearSVC Baseline
print("Fitting LinearSVC Baseline...")
svc = LinearSVC(C=1.0, max_iter=5000, random_state=42)
svc.fit(X_train, y_train)

svc_test_preds = svc.predict(X_test)
svc_chal_preds = svc.predict(X_chal)

res_svc_test = evaluate_preds(y_test, svc_test_preds, 'test', 'LinearSVC')
res_svc_chal = evaluate_preds(y_chal, svc_chal_preds, 'challenge', 'LinearSVC')

# 2. TorchMLP Baseline
print("\nTraining TorchMLP Baseline...")
mlp = train_torch_mlp(X_train, y_train, X_test, y_test, num_classes=len(EMOTION_LABELS))
mlp_test_preds = predict_mlp(mlp, X_test)
mlp_chal_preds = predict_mlp(mlp, X_chal)

res_mlp_test = evaluate_preds(y_test, mlp_test_preds, 'test', 'TorchMLP')
res_mlp_chal = evaluate_preds(y_chal, mlp_chal_preds, 'challenge', 'TorchMLP')

# Display Per-class Comparison
baseline_summary = pd.DataFrame([
    {
        'Model': r['model'],
        'Split': r['split'],
        'Macro F1': f"{r['macro_f1']:.4f}",
        'Weighted F1': f"{r['weighted_f1']:.4f}",
        **{f"{cls} F1": f"{r['per_class_f1'][cls]:.3f}" for cls in EMOTION_LABELS}
    }
    for r in [res_svc_test, res_svc_chal, res_mlp_test, res_mlp_chal]
])
print("\n--- Baseline Results Summary ---")
print(baseline_summary)

# Save baseline results
with open(os.path.join(PHASE13_DIR, 'baseline_results.json'), 'w') as f:
    json.dump([res_svc_test, res_svc_chal, res_mlp_test, res_mlp_chal], f, indent=2)


In [ ]:
# 11. Baseline Confusion Matrix Visualization (Test vs Challenge Sets)
fig, axes = plt.subplots(2, 2, figsize=(16.0, 13.0), dpi=300)

cm_svc_test = confusion_matrix(y_test, svc_test_preds)
cm_mlp_test = confusion_matrix(y_test, mlp_test_preds)
cm_svc_chal = confusion_matrix(y_chal, svc_chal_preds)
cm_mlp_chal = confusion_matrix(y_chal, mlp_chal_preds)

models_eval = [
    (axes[0, 0], cm_svc_test, f"LinearSVC Baseline (Test Set, N={len(y_test):,})", "Blues", res_svc_test['macro_f1']),
    (axes[0, 1], cm_mlp_test, f"TorchMLP Baseline (Test Set, N={len(y_test):,})", "Purples", res_mlp_test['macro_f1']),
    (axes[1, 0], cm_svc_chal, f"LinearSVC Baseline (Challenge Set, N={len(y_chal):,})", "Blues", res_svc_chal['macro_f1']),
    (axes[1, 1], cm_mlp_chal, f"TorchMLP Baseline (Challenge Set, N={len(y_chal):,})", "Purples", res_mlp_chal['macro_f1'])
]

for ax, cm, title_prefix, cmap, f1_val in models_eval:
    # Compute row-normalized percentages (Class Recall %)
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_norm = (cm / row_sums) * 100

    # Dual annotation matrix: Recall % on top, count below
    annot_matrix = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot_matrix[i, j] = f"{cm_norm[i, j]:.1f}%\n({cm[i, j]:,})"

    sns.heatmap(
        cm_norm,
        annot=annot_matrix,
        fmt="",
        cmap=cmap,
        vmin=0,
        vmax=100,
        square=True,
        linewidths=0.8,
        linecolor='white',
        xticklabels=EMOTION_LABELS,
        yticklabels=EMOTION_LABELS,
        cbar_kws={'label': 'Class Recall Rate (%)', 'shrink': 0.8},
        annot_kws={'size': 7.0},
        ax=ax
    )
    ax.set_title(f"{title_prefix} — Macro F1: {f1_val:.4f}", fontweight='bold', pad=10)
    ax.set_xlabel("Predicted Emotion Label", fontweight='bold')
    ax.set_ylabel("True Emotion Label", fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout(w_pad=3.0, h_pad=3.0)
plt.savefig(os.path.join(PHASE13_DIR, 'baseline_confusion_matrices.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(PHASE13_DIR, 'baseline_confusion_matrices.pdf'), bbox_inches='tight')
plt.show()


# Per-class recall breakdown computed directly from confusion matrices (Test vs Challenge)
recalls_df = pd.DataFrame({
    'Emotion': EMOTION_LABELS,
    'Test Support': cm_svc_test.sum(axis=1),
    'LinearSVC Test (%)': [(cm_svc_test[i, i] / cm_svc_test[i].sum() * 100) if cm_svc_test[i].sum() > 0 else 0.0 for i in range(len(EMOTION_LABELS))],
    'TorchMLP Test (%)': [(cm_mlp_test[i, i] / cm_mlp_test[i].sum() * 100) if cm_mlp_test[i].sum() > 0 else 0.0 for i in range(len(EMOTION_LABELS))],
    'Chal Support': cm_svc_chal.sum(axis=1),
    'LinearSVC Chal (%)': [(cm_svc_chal[i, i] / cm_svc_chal[i].sum() * 100) if cm_svc_chal[i].sum() > 0 else 0.0 for i in range(len(EMOTION_LABELS))],
    'TorchMLP Chal (%)': [(cm_mlp_chal[i, i] / cm_mlp_chal[i].sum() * 100) if cm_mlp_chal[i].sum() > 0 else 0.0 for i in range(len(EMOTION_LABELS))]
})
print("\n--- Per-Class Recall Breakdown: Test Set vs Challenge Set ---")
print(recalls_df.round(1).to_string(index=False))


## 5. Out-of-Fold (OOF) Linear SVM on Training Set

To rigorously evaluate baseline model behavior and tail-class collapse **purely on the training data** without touching the test or challenge holdouts, we execute 5-fold Stratified Cross-Validation on $(X_{\text{train}}, y_{\text{train}})$.

- **Zero Leakage**: Every training sample receives an Out-of-Fold (OOF) prediction generated by a `LinearSVC` model trained strictly on the other 4 folds.
- **Stratification**: Preserves the extreme 139.2 : 1 class distribution across all folds, allocating exactly 7–8 samples of the rare tail classes (`Fear`: 38, `Surprise`: 38) to each validation fold.
- **Decision Margin**: Captures out-of-fold continuous decision function margins across all 7 classes to inspect separation quality.


In [ ]:
# 12. 5-Fold Stratified Out-of-Fold (OOF) Linear SVM on Training Set
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support

print("=== Performing 5-Fold Stratified OOF Cross-Validation on Training Data ===")
print(f"Training pool size: {len(X_train):,} samples across {len(EMOTION_LABELS)} emotion classes")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(y_train), dtype=int)
oof_decision = np.zeros((len(y_train), len(EMOTION_LABELS)), dtype=np.float32)
infold_decision_sum = np.zeros((len(y_train), len(EMOTION_LABELS)), dtype=np.float32)
infold_counts = np.zeros(len(y_train), dtype=int)
fold_metrics = []
first_fold_data = None

for fold, (train_idx_cv, val_idx_cv) in enumerate(skf.split(X_train, y_train), 1):
    X_tr_cv, y_tr_cv = X_train[train_idx_cv], y_train[train_idx_cv]
    X_val_cv, y_val_cv = X_train[val_idx_cv], y_train[val_idx_cv]
    
    clf_fold = LinearSVC(C=1.0, max_iter=5000, random_state=42)
    clf_fold.fit(X_tr_cv, y_tr_cv)
    
    preds_val = clf_fold.predict(X_val_cv)
    oof_preds[val_idx_cv] = preds_val
    oof_decision[val_idx_cv] = clf_fold.decision_function(X_val_cv)
    
    # Accumulate in-fold training scores
    infold_decision_sum[train_idx_cv] += clf_fold.decision_function(X_tr_cv)
    infold_counts[train_idx_cv] += 1
    
    if fold == 1:
        first_fold_data = (clf_fold, train_idx_cv, val_idx_cv)
    
    f_f1 = f1_score(y_val_cv, preds_val, average='macro', zero_division=0)
    f_wf1 = f1_score(y_val_cv, preds_val, average='weighted', zero_division=0)
    f_acc = accuracy_score(y_val_cv, preds_val)
    fold_metrics.append({
        'fold': fold,
        'val_size': len(val_idx_cv),
        'macro_f1': float(f_f1),
        'weighted_f1': float(f_wf1),
        'accuracy': float(f_acc)
    })
    print(f"  Fold {fold}/5: Val Samples = {len(val_idx_cv):4d} | Macro F1 = {f_f1:.4f} | Weighted F1 = {f_wf1:.4f} | Acc = {f_acc:.4f}")

# Compute in-fold averaged decision scores and signed margin gaps
infold_decision = infold_decision_sum / np.maximum(infold_counts[:, None], 1)

oof_s_y = oof_decision[np.arange(len(y_train)), y_train]
oof_max_other = np.max(oof_decision + np.eye(len(EMOTION_LABELS))[y_train] * -1e9, axis=1)
oof_margin_gap = oof_s_y - oof_max_other

infold_s_y = infold_decision[np.arange(len(y_train)), y_train]
infold_max_other = np.max(infold_decision + np.eye(len(EMOTION_LABELS))[y_train] * -1e9, axis=1)
infold_margin_gap = infold_s_y - infold_max_other

# Compute overall OOF metrics on the full training pool
oof_macro_f1 = f1_score(y_train, oof_preds, average='macro', zero_division=0)
oof_weighted_f1 = f1_score(y_train, oof_preds, average='weighted', zero_division=0)
oof_acc = accuracy_score(y_train, oof_preds)

print(f"\n" + "="*60)
print(f"=== Overall OOF Linear SVM Performance (Training Data N={len(y_train):,}) ===")
print(f"  OOF Macro F1:    {oof_macro_f1:.4f}")
print(f"  OOF Weighted F1: {oof_weighted_f1:.4f}")
print(f"  OOF Accuracy:    {oof_acc:.4f} ({oof_acc*100:.2f}%)")
print("="*60)

# Per-class precision, recall, F1
prec, rec, f1, supp = precision_recall_fscore_support(y_train, oof_preds, labels=range(len(EMOTION_LABELS)), zero_division=0)
oof_per_class_df = pd.DataFrame({
    'Emotion': EMOTION_LABELS,
    'Train Support': supp,
    'Class %': (supp / len(y_train) * 100).round(2),
    'Precision (%)': (prec * 100).round(1),
    'Recall (%)': (rec * 100).round(1),
    'F1-Score': f1.round(4)
})
print("\n--- Per-Class OOF Breakdown ---")
print(oof_per_class_df.to_string(index=False))

# Save OOF results to JSON
oof_results = {
    'model': 'LinearSVC_OOF',
    'dataset': 'UAReviews_Train_Only',
    'n_samples': int(len(y_train)),
    'cv_folds': 5,
    'fold_metrics': fold_metrics,
    'oof_macro_f1': float(oof_macro_f1),
    'oof_weighted_f1': float(oof_weighted_f1),
    'oof_accuracy': float(oof_acc),
    'per_class': oof_per_class_df.to_dict(orient='records')
}
with open(os.path.join(PHASE13_DIR, 'oof_linearsvc_results.json'), 'w') as f:
    json.dump(oof_results, f, indent=2)
print(f"\nSaved OOF results to: {os.path.join(PHASE13_DIR, 'oof_linearsvc_results.json')}")


In [ ]:
# 13. OOF Confusion Matrix Visualization & Comparison
cm_oof = confusion_matrix(y_train, oof_preds)
cm_oof_norm = cm_oof.astype('float') / cm_oof.sum(axis=1)[:, np.newaxis] * 100

fig, ax = plt.subplots(figsize=(8.8, 7.0), dpi=300)
annot_matrix = np.array([
    [f"{pct:.1f}%\n({cnt:,})" for pct, cnt in zip(row_norm, row_cnt)]
    for row_norm, row_cnt in zip(cm_oof_norm, cm_oof)
])

sns.heatmap(
    cm_oof_norm,
    annot=annot_matrix,
    fmt="",
    cmap="Blues",
    vmin=0,
    vmax=100,
    square=True,
    linewidths=0.8,
    linecolor='white',
    xticklabels=EMOTION_LABELS,
    yticklabels=EMOTION_LABELS,
    cbar_kws={'label': 'Class Recall Rate (%)', 'shrink': 0.8},
    annot_kws={'size': 7.5},
    ax=ax
)
ax.set_title(
    f"Out-of-Fold (OOF) LinearSVC Baseline — Training Data Only (N={len(y_train):,})\n"
    f"OOF Macro F1: {oof_macro_f1:.4f} | Weighted F1: {oof_weighted_f1:.4f} | Accuracy: {oof_acc*100:.2f}%",
    fontweight='bold', pad=12
)
ax.set_xlabel("Predicted Emotion Label", fontweight='bold')
ax.set_ylabel("True Emotion Label", fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(PHASE13_DIR, 'oof_linearsvc_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(PHASE13_DIR, 'oof_linearsvc_confusion_matrix.pdf'), bbox_inches='tight')
plt.show()

# Comparison: OOF on Training Set vs Holdout Test Set
test_macro = res_svc_test['macro_f1']
test_acc = res_svc_test['accuracy']
test_per_class = res_svc_test['per_class_f1']

comp_df = pd.DataFrame({
    'Emotion': EMOTION_LABELS,
    'Train Support': supp,
    'OOF F1 (Train N=8,106)': oof_per_class_df['F1-Score'],
    'Test F1 (Holdout N=1,737)': [test_per_class[e] for e in EMOTION_LABELS]
})
print("\n--- Comparison: OOF (Training Data) vs Holdout (Test Set) ---")
print(comp_df.to_string(index=False))
print(f"\nSummary Comparison:")
print(f"  OOF Macro F1:  {oof_macro_f1:.4f}  vs  Test Macro F1:  {test_macro:.4f}")
print(f"  OOF Accuracy:  {oof_acc:.4f}  vs  Test Accuracy:  {test_acc:.4f}")


In [ ]:
# 14. Option 1: 2D Decision-Space Separability Plot (In-Fold vs OOF)
# Projects points onto decision scores: X = s_Happiness, Y = s_Tail
# Decision boundary is the exact line y = x (where s_Tail = s_Happiness)
clf1, tr_idx1, val_idx1 = first_fold_data
s_tr = clf1.decision_function(X_train[tr_idx1])
s_val = clf1.decision_function(X_train[val_idx1])
y_tr = y_train[tr_idx1]
y_val = y_train[val_idx1]

targets = [
    ('Fear', 6, EMOTION_PALETTE['Fear']),
    ('Surprise', 5, EMOTION_PALETTE['Surprise']),
    ('Anger', 1, EMOTION_PALETTE['Anger'])
]

fig, axes = plt.subplots(1, 3, figsize=(18.0, 5.6), dpi=300)

for idx, (t_name, t_cls, t_color) in enumerate(targets):
    ax = axes[idx]
    
    tr_head_x = s_tr[y_tr == 0, 0]
    tr_head_y = s_tr[y_tr == 0, t_cls]
    tr_tail_x = s_tr[y_tr == t_cls, 0]
    tr_tail_y = s_tr[y_tr == t_cls, t_cls]
    
    val_head_x = s_val[y_val == 0, 0]
    val_head_y = s_val[y_val == 0, t_cls]
    val_tail_x = s_val[y_val == t_cls, 0]
    val_tail_y = s_val[y_val == t_cls, t_cls]
    
    all_x = np.concatenate([tr_head_x, tr_tail_x, val_head_x, val_tail_x])
    all_y = np.concatenate([tr_head_y, tr_tail_y, val_head_y, val_tail_y])
    lim_min = min(all_x.min(), all_y.min()) - 0.5
    lim_max = max(all_x.max(), all_y.max()) + 0.5
    grid_pts = np.linspace(lim_min, lim_max, 100)
    
    # Shaded decision regions
    ax.fill_between(grid_pts, grid_pts, lim_max, color=t_color, alpha=0.07, label=f'Predicted {t_name} Region (s_{t_name} > s_Happy)')
    ax.fill_between(grid_pts, lim_min, grid_pts, color=EMOTION_PALETTE['Happiness'], alpha=0.04, label=f'Predicted Happiness Region (s_Happy > s_{t_name})')
    
    # Decision line y = x
    ax.plot(grid_pts, grid_pts, color='black', linestyle='--', linewidth=1.5, label=f'Decision Boundary ($s_{{{t_name}}} = s_{{Happy}}$)')
    
    # In-fold training points (faint/open markers)
    ax.scatter(tr_head_x, tr_head_y, color=EMOTION_PALETTE['Happiness'], alpha=0.15, s=12, marker='o', rasterized=True, label='In-Fold Happiness (Train)')
    ax.scatter(tr_tail_x, tr_tail_y, edgecolor=t_color, facecolors='none', alpha=0.85, s=38, linewidth=1.3, marker='o', label=f'In-Fold {t_name} (Train)')
    
    # OOF validation points (solid/distinct markers)
    ax.scatter(val_head_x, val_head_y, color=EMOTION_PALETTE['Happiness'], alpha=0.35, s=18, marker='^', rasterized=True, label='OOF Happiness (Val)')
    ax.scatter(val_tail_x, val_tail_y, color=t_color, edgecolor='black', alpha=0.95, s=70, linewidth=1.4, marker='*', label=f'OOF {t_name} (Val)')
    
    ax.set_xlim(lim_min, lim_max)
    ax.set_ylim(lim_min, lim_max)
    ax.set_xlabel('Happiness Decision Score ($s_{Happiness}$)', fontweight='bold')
    ax.set_ylabel(f'{t_name} Decision Score ($s_{{{t_name}}}$)', fontweight='bold')
    ax.set_title(f"Separability: Happiness vs {t_name}\n(OOF {t_name} Collapsing Below Boundary into Happiness)", fontweight='bold', pad=10)
    ax.legend(loc='lower right', fontsize=7.2, frameon=True, framealpha=0.92)

plt.tight_layout()
plt.savefig(os.path.join(PHASE13_DIR, 'oof_separability_decision_space.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(PHASE13_DIR, 'oof_separability_decision_space.pdf'), bbox_inches='tight')
plt.show()


In [ ]:
# 15. Option 2: 1D Signed Margin Gap Distribution (Train vs OOF Generalization Gap)
# Delta(x) = s_true - max_{c != true} s_c
# Delta > 0: Correct classification | Delta < 0: Margin violation/error
rows = []
for i in range(len(y_train)):
    emo = EMOTION_LABELS[y_train[i]]
    rows.append({'Emotion': emo, 'Evaluation': 'In-Fold (Train)', 'Margin Gap (Δ)': float(infold_margin_gap[i])})
    rows.append({'Emotion': emo, 'Evaluation': 'Out-of-Fold (OOF)', 'Margin Gap (Δ)': float(oof_margin_gap[i])})

df_margins = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(13.0, 6.0), dpi=300)

sns.boxplot(
    data=df_margins,
    x='Emotion',
    y='Margin Gap (Δ)',
    hue='Evaluation',
    order=EMOTION_LABELS,
    palette={'In-Fold (Train)': '#345995', 'Out-of-Fold (OOF)': '#d95f02'},
    ax=ax,
    fliersize=1.8,
    linewidth=1.1,
    whis=1.5
)

ax.axhline(0, color='red', linestyle='--', linewidth=1.5, zorder=10)
ax.text(6.35, 0.08, 'Decision Boundary (Δ = 0)', color='red', fontweight='bold', fontsize=8.5, va='bottom', ha='right')

min_y = df_margins['Margin Gap (Δ)'].min() - 0.5
ax.axhspan(min_y, 0, color='red', alpha=0.06)
ax.text(0.02, 0.08, 'Misclassification Zone (Δ < 0)', transform=ax.get_yaxis_transform(), color='#b2182b', fontweight='bold', fontsize=8.5)

ax.set_title("Linear SVM Signed Margin Gap Distribution (In-Fold Train vs Out-of-Fold OOF)\nVisualizes Margin Generalization Gap & Complete Collapse on Tail Classes (Fear & Surprise)", fontweight='bold', pad=12)
ax.set_xlabel("Emotion Category (Ordered Head to Tail)", fontweight='bold')
ax.set_ylabel("Signed Margin Gap: $\\Delta(x) = s_{y} - \\max_{c \\ne y} s_c$", fontweight='bold')
ax.legend(loc='upper right', frameon=True, framealpha=0.92)

plt.tight_layout()
plt.savefig(os.path.join(PHASE13_DIR, 'oof_margin_distributions.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(PHASE13_DIR, 'oof_margin_distributions.pdf'), bbox_inches='tight')
plt.show()
